In [ ]:
# --- Cell 1: Install TRL (offline auto-detect without internet) ---
import subprocess, sys
from pathlib import Path
try:
    import trl
    print(f"TRL already installed: {trl.__version__}")
except ImportError:
    # Auto-find the uploaded TRL wheel directory in Kaggle inputs
    trl_wheel_dir = "/kaggle/input/trl-offline"  # fallback
    input_dir = Path("/kaggle/input")
    if input_dir.exists():
        for p in input_dir.rglob("trl*none-any.whl"):
            trl_wheel_dir = str(p.parent)
            break

    print(f"Installing TRL offline from: {trl_wheel_dir}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--find-links", trl_wheel_dir, "trl", "--quiet"],
        check=True,
    )
    import trl
    print(f"Installed TRL: {trl.__version__}")

In [ ]:
# --- Install bitsandbytes from offline wheels (Kaggle: internet off on accelerator) ---
import subprocess
import sys
from pathlib import Path


def _find_bitsandbytes_findlinks() -> str | None:
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    for whl in root.rglob("bitsandbytes*.whl"):
        return str(whl.parent)
    return None


try:
    import bitsandbytes  # noqa: F401

    print("bitsandbytes already importable.")
except ImportError:
    links = _find_bitsandbytes_findlinks()
    if not links:
        print(
            "WARNING: No bitsandbytes wheel under /kaggle/input. "
            "Upload repo folder `bnb_offline/` as a Kaggle Dataset and attach it. "
            "Next cell will fall back to optim=adamw_torch_fused if import still fails."
        )
    else:
        print(f"Installing bitsandbytes (offline) from: {links}")
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--no-index",
                "--find-links",
                links,
                "bitsandbytes",
                "--no-deps",
                "--quiet",
            ],
            check=True,
        )
        print("bitsandbytes pip install finished.")


In [ ]:
# --- Cell 2: Config + Imports ---
import site
import sys
import os
from pathlib import Path

if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

# CUTLASS path setup (from NVIDIA utility script)
candidate_cutlass_paths = [
    Path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"),
    Path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages/"),
]
for p in candidate_cutlass_paths:
    if p.exists():
        site.addsitedir(str(p))
        print(f"Added cutlass path: {p}")
        break

import pandas as pd
import torch
import kagglehub
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    import mamba_ssm  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "mamba_ssm import failed. Run TRL + bitsandbytes cells, restart kernel, then re-run from Config."
    ) from exc

# ═══════════════════════════════════════════════════════════
# TRAINING CONFIG — Run B (mixed CoT + bare, flash attention, packing)
# ═══════════════════════════════════════════════════════════
OUTPUT_DIR = "/kaggle/working"
FINAL_ADAPTER_DIR = f"{OUTPUT_DIR}/final_adapter"
MODEL_ID = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
LORA_RANK = 32

# Training knobs
MAX_TRAIN_SAMPLES = None        # e.g. 256 for a quick run
NUM_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
SAVE_STEPS = 250
EVAL_STEPS = 250
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 2048           # increased for CoT traces (longer completions)
EVAL_MAX_NEW_TOKENS = 2048
LR_SCHEDULER_TYPE = "cosine"
WARMUP_RATIO = 0.05

# CoT traces CSV (exported from task-cot.ipynb). Set to None to skip CoT mixing.
COT_TRACES_FILE = "cot_traces_verified.csv"
try:
    import bitsandbytes as _bnb
    _bnb.optim.Adam8bit  # force CUDA init — fails fast if binary missing
    OPTIM = "adamw_8bit"
    print("Using optim=adamw_8bit (bitsandbytes", getattr(_bnb, "__version__", "?"), ")")
except Exception:
    OPTIM = "adamw_torch_fused"
    print("bitsandbytes unavailable or CUDA mismatch — using optim=adamw_torch_fused")

# ═══════════════════════════════════════════════════════════
# EVAL CONFIG
# ═══════════════════════════════════════════════════════════
USE_EVAL = True   # False for final submission → train on all 9,500 rows
SPLIT_DATASET_SLUG = "your-username/nemotron-split-data"  # ← UPDATE after uploading

# Suffix — must match nvidia-nemotron-metric.ipynb exactly
BOXED_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())


In [ ]:
# --- Cell 3: Data loading (eval-split aware) ---
from pathlib import Path

def resolve_train_csv() -> Path:
    """Find the competition train.csv on Kaggle or locally."""
    candidates = [
        Path("/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"),
        Path("/kaggle/input/competitions/nvidia-nemotron-3-reasoning-challenge/train.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    found = (
        list(Path("/kaggle/input").rglob("train.csv"))
        if Path("/kaggle/input").exists()
        else []
    )
    if found:
        return found[0]
    local = Path("train.csv")
    if local.exists():
        return local.resolve()
    raise FileNotFoundError(
        "train.csv not found. Add the competition dataset as a Kaggle input."
    )

if USE_EVAL:
    # Auto-find the uploaded split dataset in Kaggle inputs
    split_root = None
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob("train_9000.csv"):
            split_root = p.parent
            break
    
    if not split_root:
        # Fallback to local datasets folder
        local_split_root = Path("datasets")
        if local_split_root.exists():
            split_root = local_split_root
        else:
            raise FileNotFoundError(
                "Split data not found. Upload datasets/ folder as a Kaggle dataset or ensure local files exist."
            )

    train_df = pd.read_csv(split_root / "train_9000.csv")
    eval_df = pd.read_csv(split_root / "eval_500.csv")
    print(f"\u2705 Eval mode: Train={len(train_df)}, Eval={len(eval_df)} (from {split_root})")
else:
    # Final submission: use ALL competition data
    train_df = pd.read_csv(resolve_train_csv())
    eval_df = None
    print(f"\U0001f3c1 Final submission mode: Training on ALL {len(train_df)} rows (no eval)")

# Apply sample limit if set
if MAX_TRAIN_SAMPLES is not None:
    train_df = train_df.head(int(MAX_TRAIN_SAMPLES)).copy()
    print(f"\u26a0\ufe0f Truncated to {len(train_df)} samples (MAX_TRAIN_SAMPLES={MAX_TRAIN_SAMPLES})")

print(f"Training rows: {len(train_df)}")

In [ ]:
# --- Cell 4: Tokenizer + data formatting + CoT mixing ---
import os
from pathlib import Path
from glob import glob as _glob

def resolve_model_path() -> str:
    """Find the Nemotron model on Kaggle inputs or download via kagglehub."""
    direct_candidates = [
        Path("/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default"),
        Path("/kaggle/input/nemotron-3-nano/transformers/default/1"),
    ]
    for path in direct_candidates:
        if path.exists():
            print(f"Found model in Kaggle inputs: {path}")
            return str(path)

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for config_path in input_root.rglob("config.json"):
            candidate = config_path.parent
            candidate_text = str(candidate).lower()
            if "nemotron" not in candidate_text:
                continue
            if not (candidate / "tokenizer_config.json").exists():
                continue
            print(f"Found model in Kaggle inputs: {candidate}")
            return str(candidate)

    print("Downloading model via kagglehub (requires internet)...")
    return kagglehub.model_download(MODEL_ID)

MODEL_PATH = resolve_model_path()

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

def format_prompt(prompt: str) -> str:
    user_content = str(prompt).strip() + BOXED_SUFFIX
    messages = [{"role": "user", "content": user_content}]
    kwargs = dict(tokenize=False, add_generation_prompt=True)
    try:
        return tokenizer.apply_chat_template(messages, **kwargs, enable_thinking=True)
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kwargs)

def format_completion_bare(answer: str) -> str:
    return f"\\boxed{{{str(answer).strip()}}}"

def format_completion_cot(cot_response: str, answer: str) -> str:
    """Wrap CoT reasoning in <think> tags + boxed answer (matches Nemotron thinking format)."""
    reasoning = str(cot_response).strip()
    gold = str(answer).strip()
    return f"\n{reasoning}\n</think>\n\\boxed{{{gold}}}"

# --- Build bare-answer training rows (8500 from train_9000 minus any CoT-replaced rows) ---
prompts = []
completions = []

# --- Load CoT traces if available ---
cot_df = None
cot_row_ids = set()
if COT_TRACES_FILE:
    cot_candidates = []
    if Path("/kaggle/input").exists():
        cot_candidates = sorted(Path("/kaggle/input").rglob(COT_TRACES_FILE))
    if not cot_candidates:
        local_cot = Path("benchmark_outputs") / COT_TRACES_FILE
        if local_cot.exists():
            cot_candidates = [local_cot]
        local_cot2 = Path("datasets") / COT_TRACES_FILE
        if local_cot2.exists():
            cot_candidates.append(local_cot2)

    if cot_candidates:
        cot_df = pd.read_csv(cot_candidates[0])
        print(f"Loaded CoT traces: {cot_candidates[0]} ({len(cot_df)} verified rows)")

        # Match CoT rows to train_df by question text (row_id may differ between splits)
        cot_questions = set(cot_df["question"].astype(str).str.strip())

        # Add CoT rows as training examples
        cot_count = 0
        for _, crow in cot_df.iterrows():
            q = str(crow["question"]).strip()
            p = format_prompt(q)
            c = format_completion_cot(crow["cot_response"], crow["gold_answer"])
            prompts.append(p)
            completions.append(c)
            cot_count += 1
            cot_row_ids.add(q)
        print(f"Added {cot_count} CoT-formatted training rows")
    else:
        print(f"No CoT traces found ({COT_TRACES_FILE}). Training with bare answers only.")

# Add remaining bare-answer rows (skip any that have CoT replacements)
bare_count = 0
for row in train_df.itertuples(index=False):
    q = str(row.prompt).strip()
    if q in cot_row_ids:
        continue
    prompts.append(format_prompt(row.prompt))
    completions.append(format_completion_bare(row.answer))
    bare_count += 1

train_ds = Dataset.from_dict({"prompt": prompts, "completion": completions})

eval_ds = None
if eval_df is not None:
    _ep = [format_prompt(r.prompt) for r in eval_df.itertuples(index=False)]
    _ec = [format_completion_bare(r.answer) for r in eval_df.itertuples(index=False)]
    eval_ds = Dataset.from_dict({"prompt": _ep, "completion": _ec})
    print(f"Trainer eval dataset: {len(eval_ds)} examples")

print(f"\nTraining dataset: {len(train_ds)} examples ({bare_count} bare + {len(train_ds)-bare_count} CoT)")
print(f"\n--- Sample prompt (first 500 chars) ---")
print(train_ds[0]["prompt"][:500])
print(f"\n--- Sample completion (first 300 chars) ---")
print(train_ds[0]["completion"][:300])

In [ ]:
# --- Cell 7: Load 30B model + attach LoRA (rank 32) ---
# flash_attention_2 prevents cross-contamination between packed samples
_attn_impl = "flash_attention_2"
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map="auto",
        trust_remote_code=True,
        dtype=torch.bfloat16,
        attn_implementation=_attn_impl,
    )
    print(f"Loaded model with attn_implementation={_attn_impl}")
except Exception as e:
    print(f"flash_attention_2 unavailable ({e}), falling back to default attention")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map="auto",
        trust_remote_code=True,
        dtype=torch.bfloat16,
    )

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# --- Cell 8: Training (SFTTrainer with response-only loss) ---
from trl import SFTTrainer, SFTConfig
import shutil
from pathlib import Path
import subprocess

os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)

# ─── SFTConfig (Run A-prime: packing + optional best checkpoint on eval_loss) ───
_sft_common = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_ratio=WARMUP_RATIO,
    optim=OPTIM,
    logging_steps=50,
    ignore_data_skip=True,
    bf16=True,
    report_to="none",
    dataset_text_field=None,
    completion_only_loss=True,
    max_length=MAX_SEQ_LENGTH,
    packing=True,
)

if eval_ds is not None:
    sft_config = SFTConfig(
        **_sft_common,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        per_device_eval_batch_size=1,
        save_strategy="best",
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        load_best_model_at_end=True,
        save_total_limit=1,
    )
else:
    sft_config = SFTConfig(
        **_sft_common,
        eval_strategy="no",
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=1,
    )

# ─── Create trainer ───
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

# ─── Fix Kaggle read-only Triton binary permissions ───
triton_bin_candidates = [
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin",
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin",
]
ro_bin_dir = next((path for path in triton_bin_candidates if os.path.exists(path)), None)
rw_bin_dir = "/kaggle/working/triton_bin"
if ro_bin_dir:
    os.makedirs(rw_bin_dir, exist_ok=True)
    for f in os.listdir(ro_bin_dir):
        src = os.path.join(ro_bin_dir, f)
        dst = os.path.join(rw_bin_dir, f)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
        os.chmod(dst, 0o777)

    orig_popen = subprocess.Popen

    def patched_popen(*args, **kwargs):
        if args and isinstance(args[0], list) and isinstance(args[0][0], str):
            if args[0][0].startswith(ro_bin_dir):
                args[0][0] = args[0][0].replace(ro_bin_dir, rw_bin_dir)
        return orig_popen(*args, **kwargs)

    subprocess.Popen = patched_popen

# ─── Print config summary ───
_has_cot = len(train_ds) > bare_count if 'bare_count' in dir() else False
_run_name = "Run B (mixed CoT + bare)" if _has_cot else "Run A-prime (bare only)"
print("=" * 60)
print(f"TRAINING CONFIG ({_run_name})")
print("=" * 60)
print(f"  Epochs:         {NUM_EPOCHS}")
print(f"  LR:             {LEARNING_RATE}")
print(f"  LR schedule:    {LR_SCHEDULER_TYPE}")
print(f"  Warmup ratio:   {WARMUP_RATIO}")
print(f"  Optimizer:      {OPTIM}")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")
print(f"  Post-train eval: max_new_tokens={EVAL_MAX_NEW_TOKENS}")
print(f"  Batch size:     {PER_DEVICE_TRAIN_BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} grad accum")
print(f"  Response-only:  {getattr(sft_config, 'completion_only_loss', True)}")
print(f"  Packing:        {getattr(sft_config, 'packing', False)}")
print(f"  Flash attn:     {_attn_impl if '_attn_impl' in dir() else 'unknown'}")
print(f"  Train samples:  {len(train_ds)} (CoT: {len(train_ds) - bare_count if 'bare_count' in dir() else 0})")
print(f"  Save:           {'best@eval_loss' if eval_ds is not None else 'steps=' + str(SAVE_STEPS)}, total_limit=1")
print("=" * 60)

# Resume from prior session (attach previous output / dataset with checkpoint-*)
resume_path = None
_inp = Path("/kaggle/input")
if _inp.exists():
    for _cp in sorted(_inp.rglob("checkpoint-*"), reverse=True):
        if (_cp / "trainer_state.json").exists():
            _dest = Path("/kaggle/working") / _cp.name
            if not _dest.exists():
                shutil.copytree(str(_cp), str(_dest))
            resume_path = str(_dest)
            print("Resuming from:", resume_path)
            break
if resume_path is None:
    print("No checkpoint in /kaggle/input - training from scratch")

trainer.train(resume_from_checkpoint=resume_path)


In [ ]:
import shutil
# --- Cell 9: Save trained adapter weights ---
from pathlib import Path

# Free disk: remove training checkpoint(s) before saving final adapter
import glob as _gl
for _ckpt in _gl.glob(f"{OUTPUT_DIR}/checkpoint-*"):
    shutil.rmtree(_ckpt, ignore_errors=True)
    print(f"Removed {_ckpt}")

final_adapter_dir = Path(FINAL_ADAPTER_DIR)
final_adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)
print("Saved adapter + tokenizer under", final_adapter_dir)


In [ ]:
# --- Cell 10: Zip adapter files for Kaggle submission ---
import subprocess
from pathlib import Path

out = Path(OUTPUT_DIR)
adapter_dir = Path(FINAL_ADAPTER_DIR)
submission_zip = out / "submission.zip"
cfg = adapter_dir / "adapter_config.json"
weights = adapter_dir / "adapter_model.safetensors"
if not weights.exists():
    weights = adapter_dir / "adapter_model.bin"
if not cfg.exists() or not weights.exists():
    raise FileNotFoundError(f"Missing adapter files. Expected {cfg} and a weights file.")

if submission_zip.exists():
    submission_zip.unlink()

subprocess.run([
    "zip",
    "-j",
    str(submission_zip),
    str(cfg),
    str(weights),
], check=True)
print("Wrote", submission_zip)

In [ ]:

# --- Cell 11: Evaluation on 500 held-out rows ---
import json
import re
import math
from pathlib import Path

if eval_df is not None:
    print("=" * 60)
    print(f"RUNNING EVALUATION on {len(eval_df)} held-out rows")
    print("=" * 60)

    # ─── Family classification (keyword-based) ───
    def classify_family(prompt_text: str) -> str:
        p = str(prompt_text).lower()
        if "bit manipulation" in p:
            return "bit_manipulation"
        if "gravitational constant" in p:
            return "gravity"
        if "encryption rules" in p or "encryption" in p:
            return "encryption"
        if "numeral system" in p:
            return "numeral"
        if "unit conversion" in p or "converted" in p:
            return "unit_conversion"
        if "transformation rules" in p or "equation" in p or "operator" in p:
            return "equations"
        return "unknown"

    # ─── Answer extraction (matches nvidia-nemotron-metric.ipynb) ───
    def extract_final_answer(text):
        if text is None:
            return "NOT_FOUND"
        matches = re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
        if matches:
            non_empty = [m.strip() for m in matches if m.strip()]
            if non_empty:
                return non_empty[-1]
            return matches[-1].strip()
        patterns = [
            r'The final answer is:\s*([^\n]+)',
            r'Final answer is:\s*([^\n]+)',
            r'Final answer\s*[:：]\s*([^\n]+)',
            r'final answer\s*[:：]\s*([^\n]+)',
        ]
        for pattern in patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            if matches:
                return matches[-1].strip()
        matches = re.findall(r'-?\d+(?:\.\d+)?', text)
        if matches:
            return matches[-1]
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        return lines[-1] if lines else "NOT_FOUND"

    # ─── Verification (matches nvidia-nemotron-metric.ipynb) ───
    def verify(stored_answer, predicted):
        stored_answer = str(stored_answer).strip()
        predicted = str(predicted).strip()
        if re.fullmatch(r'[01]+', stored_answer):
            return predicted.lower() == stored_answer.lower()
        try:
            stored_num = float(stored_answer)
            predicted_num = float(predicted)
            return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
        except Exception:
            return predicted.lower() == stored_answer.lower()

    # ─── Run inference ───
    model.eval()
    correct = 0
    total = 0
    family_stats = {}
    failures = []

    for i, row in enumerate(eval_df.itertuples(index=False)):
        # Format prompt (same as metric: user message + generation prompt)
        user_content = str(row.prompt).strip() + BOXED_SUFFIX
        messages = [{"role": "user", "content": user_content}]
        _kw = dict(tokenize=False, add_generation_prompt=True)
        try:
            prompt_text = tokenizer.apply_chat_template(messages, **_kw, enable_thinking=True)
        except TypeError:
            prompt_text = tokenizer.apply_chat_template(messages, **_kw)

        inputs = tokenizer(
            prompt_text, return_tensors="pt", truncation=True, max_length=2048
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=EVAL_MAX_NEW_TOKENS,
                temperature=1.0,
                top_p=1.0,
                do_sample=False,
            )

        # Decode only the generated (new) tokens
        generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        predicted = extract_final_answer(generated_text)
        ground_truth = str(row.answer).strip()
        is_correct = verify(ground_truth, predicted)

        # Track per-family stats
        family = classify_family(row.prompt)
        if family not in family_stats:
            family_stats[family] = {"correct": 0, "total": 0}
        family_stats[family]["total"] += 1
        if is_correct:
            correct += 1
            family_stats[family]["correct"] += 1
        elif len(failures) < 10:
            failures.append({
                "id": row.id if hasattr(row, "id") else i,
                "family": family,
                "expected": ground_truth,
                "predicted": predicted,
                "generated": generated_text[:200],
            })

        total += 1
        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{len(eval_df)}] accuracy so far: {correct}/{total} ({100*correct/total:.1f}%)")

    # ─── Print results ───
    print("\n" + "=" * 60)
    print(f"EVAL RESULTS ({total} held-out rows)")
    print("=" * 60)
    print(f"Overall accuracy: {correct}/{total} ({100*correct/total:.1f}%)")
    print("\nPer-family breakdown:")
    for fam, stats in sorted(family_stats.items()):
        pct = 100 * stats["correct"] / stats["total"] if stats["total"] > 0 else 0
        print(f"  {fam:20s}: {stats['correct']}/{stats['total']} ({pct:.1f}%)")
    if failures:
        print(f"\nSample failures (first {min(len(failures), 5)}):")
        for f in failures[:5]:
            print(f"  [{f['family']}] expected={f['expected']!r} got={f['predicted']!r}")
    print("=" * 60)

    # ─── Save eval_results.json ───
    results = {
        "overall": {
            "correct": correct,
            "total": total,
            "accuracy": correct / total if total > 0 else 0,
        },
        "per_family": family_stats,
        "failures": failures,
    }
    eval_json_path = Path(OUTPUT_DIR) / "eval_results.json"
    with open(eval_json_path, "w") as f_out:
        json.dump(results, f_out, indent=2)
    print(f"\nSaved eval results to {eval_json_path}")

else:
    print("Eval disabled (USE_EVAL=False). Skipping eval.")
